In [ ]:
import asyncio
import csv
import re
from urllib.parse import urljoin

from playwright.async_api import async_playwright

BASE_URL = "https://kolesa.group"
START_URL = "https://kolesa.group/career/job"
OUTPUT_FILE = "kolesa_vacancies.csv"


# =========================
# Utils
# =========================

def normalize_text(text):
    if not text:
        return ""
    return re.sub(r"\s+", " ", text.replace("\xa0", " ")).strip()


def normalize_list(items):
    return " | ".join([normalize_text(i) for i in items if normalize_text(i)])


def parse_salary(text):
    numbers = re.findall(r"\d[\d ]*", text)
    numbers = [int(n.replace(" ", "")) for n in numbers]

    if len(numbers) >= 2:
        return numbers[0], numbers[1]
    elif len(numbers) == 1:
        return numbers[0], None
    return None, None


# =========================
# Get вакансии (список)
# =========================

async def get_vacancies(page):
    await page.goto(START_URL)

    vacancies = {}

    tabs = page.locator('[data-test="test-tabs"]')
    await tabs.first.wait_for()

    for i in range(await tabs.count()):
        tab = tabs.nth(i)
        text = normalize_text(await tab.inner_text())

        # вытащим название категории
        category = re.sub(r"\d+$", "", text).strip()

        await tab.click()
        await page.wait_for_timeout(1000)

        cards = page.locator('[data-test="test-list-wrapper"]')

        for j in range(await cards.count()):
            card = cards.nth(j)

            href = await card.get_attribute("href")
            if not href:
                continue

            url = urljoin(BASE_URL, href)

            title = ""
            title_el = card.locator('[data-test="test-list-title"]')
            if await title_el.count() > 0:
                title = normalize_text(await title_el.inner_text())

            if url not in vacancies:
                vacancies[url] = {
                    "url": url,
                    "title": title,
                    "category": category
                }

    return list(vacancies.values())


# =========================
# Парсинг списков
# =========================

async def extract_list(page, keywords):
    for kw in keywords:
        locator = page.locator(
            f"xpath=//*[contains(text(), '{kw}')]/following::ul[1]/li"
        )
        if await locator.count() > 0:
            return await locator.all_inner_texts()
    return []


# =========================
# Парсинг страницы вакансии
# =========================

async def parse_vacancy(page, vacancy):
    try:
        await page.goto(vacancy["url"])
        await page.wait_for_selector("h1")

        # --- title ---
        title = normalize_text(await page.locator("h1").inner_text())

        # --- salary ---
        salary_text = ""
        salary_el = page.locator("text=₸")
        if await salary_el.count() > 0:
            salary_text = normalize_text(await salary_el.first.inner_text())

        salary_from, salary_to = parse_salary(salary_text)

        # --- location + experience ---
        location = ""
        experience = ""

        tags = await page.locator("li").all_inner_texts()

        for tag in tags:
            tag = normalize_text(tag)

            if "Алматы" in tag or "Астана" in tag or "Remote" in tag:
                if not location:
                    location = tag

            if "Опыт" in tag or "лет" in tag:
                if not experience:
                    experience = tag

        # --- описание компании ---
        company = ""
        about = page.locator("text=О компании")
        if await about.count() > 0:
            section = about.first.locator("xpath=following::div[1]")
            if await section.count() > 0:
                company = normalize_text(await section.inner_text())

        # --- списки ---
        tech_stack = normalize_list(await extract_list(page, ["Стек"]))
        responsibilities = normalize_list(await extract_list(page, ["Тебе предстоит"]))
        requirements = normalize_list(await extract_list(page, ["Мы ждем"]))
        nice_to_have = normalize_list(await extract_list(page, ["Будет преимуществом"]))

        vacancy.update({
            "title": title,
            "salary": salary_text,
            "salary_from": salary_from,
            "salary_to": salary_to,
            "location": location,
            "experience": experience,
            "company": company,
            "tech_stack": tech_stack,
            "responsibilities": responsibilities,
            "requirements": requirements,
            "nice_to_have": nice_to_have,
        })

    except Exception as e:
        print(f"Ошибка: {vacancy['url']} → {e}")

    return vacancy


# =========================
# Main
# =========================

async def main():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        print("Собираем список вакансий...")
        vacancies = await get_vacancies(page)

        print(f"Найдено: {len(vacancies)}")

        data = []
        for i, v in enumerate(vacancies, 1):
            print(f"[{i}/{len(vacancies)}] {v['title']}")
            parsed = await parse_vacancy(page, v)
            data.append(parsed)

        await browser.close()

        # --- CSV ---
        keys = [
            "title",
            "category",
            "url",
            "salary",
            "salary_from",
            "salary_to",
            "location",
            "experience",
            "company",
            "tech_stack",
            "responsibilities",
            "requirements",
            "nice_to_have",
        ]

        with open(OUTPUT_FILE, "w", newline="", encoding="utf-8-sig") as f:
            writer = csv.DictWriter(f, fieldnames=keys)
            writer.writeheader()
            writer.writerows(data)

        print(f"\nГотово: {OUTPUT_FILE}")


# запуск (если Jupyter)
await main()

Переходим на https://kolesa.group/career/job...
Найдено категорий: 10
--- Категория: Разработка (Ожидаем: 3) ---
   Загружено карточек: 3
--- Категория: Администрирование (Ожидаем: 1) ---
   Загружено карточек: 1
--- Категория: Data science (Ожидаем: 2) ---
   Загружено карточек: 2
--- Категория: Сервис (Ожидаем: 2) ---
   Загружено карточек: 2
--- Категория: Продажи (Ожидаем: 5) ---
   Загружено карточек: 5
--- Категория: Менеджмент (Ожидаем: 2) ---
   Загружено карточек: 2
--- Категория: Маркетинг (Ожидаем: 2) ---
   Загружено карточек: 2
--- Категория: Дизайн (Ожидаем: 1) ---
   Загружено карточек: 1
--- Категория: Бухгалтерия (Ожидаем: 1) ---
   Загружено карточек: 1
--- Категория: Другое (Ожидаем: 2) ---
   Загружено карточек: 2

Всего найдено уникальных вакансий: 21
[1/21] Middle Android-разработчик в Kolesa.kz
[2/21] Junior Android-разработчик
[3/21] Web QA (Manual + Automation)
[4/21] Специалист по ИТ/ИБ рискам
[5/21] Продуктовый аналитик в Kolesa.kz
[6/21] Machine Learning Eng

In [1]:
# new code

import asyncio
import csv
import re
from urllib.parse import urljoin

from playwright.async_api import async_playwright

BASE_URL = "https://kolesa.group"
START_URL = "https://kolesa.group/career/job"
OUTPUT_FILE = "kolesa_vacancies.csv"


# =========================
# Utils
# =========================

def normalize_text(text):
    if not text:
        return ""
    return re.sub(r"\s+", " ", text.replace("\xa0", " ")).strip()


def normalize_list(items):
    return " | ".join([normalize_text(i) for i in items if normalize_text(i)])


def parse_salary(text):
    numbers = re.findall(r"\d[\d ]*", text)
    numbers = [int(n.replace(" ", "")) for n in numbers]

    if len(numbers) >= 2:
        return numbers[0], numbers[1]
    elif len(numbers) == 1:
        return numbers[0], None
    return None, None


# =========================
# Get вакансии (список)
# =========================

async def get_vacancies(page):
    await page.goto(START_URL)

    vacancies = {}

    tabs = page.locator('[data-test="test-tabs"]')
    await tabs.first.wait_for()

    for i in range(await tabs.count()):
        tab = tabs.nth(i)
        text = normalize_text(await tab.inner_text())

        # вытащим название категории
        category = re.sub(r"\d+$", "", text).strip()

        await tab.click()
        await page.wait_for_timeout(1000)

        cards = page.locator('[data-test="test-list-wrapper"]')

        for j in range(await cards.count()):
            card = cards.nth(j)

            href = await card.get_attribute("href")
            if not href:
                continue

            url = urljoin(BASE_URL, href)

            title = ""
            title_el = card.locator('[data-test="test-list-title"]')
            if await title_el.count() > 0:
                title = normalize_text(await title_el.inner_text())

            if url not in vacancies:
                vacancies[url] = {
                    "url": url,
                    "title": title,
                    "category": category
                }

    return list(vacancies.values())


# =========================
# Парсинг списков
# =========================

async def extract_list(page, keywords):
    for kw in keywords:
        locator = page.locator(
            f"xpath=//*[contains(text(), '{kw}')]/following::ul[1]/li"
        )
        if await locator.count() > 0:
            return await locator.all_inner_texts()
    return []


# =========================
# Парсинг страницы вакансии
# =========================

async def parse_vacancy(page, vacancy):
    try:
        await page.goto(vacancy["url"])
        await page.wait_for_selector("h1")

        # --- title ---
        title = normalize_text(await page.locator("h1").inner_text())

        # --- salary ---
        salary_text = ""
        salary_el = page.locator("text=₸")
        if await salary_el.count() > 0:
            salary_text = normalize_text(await salary_el.first.inner_text())

        salary_from, salary_to = parse_salary(salary_text)

        # --- location + experience ---
        location = ""
        experience = ""

        tags = await page.locator("li").all_inner_texts()

        for tag in tags:
            tag = normalize_text(tag)

            if "Алматы" in tag or "Астана" in tag or "Remote" in tag:
                if not location:
                    location = tag

            if "Опыт" in tag or "лет" in tag:
                if not experience:
                    experience = tag

        # --- описание компании ---
        company = ""
        about = page.locator("text=О компании")
        if await about.count() > 0:
            section = about.first.locator("xpath=following::div[1]")
            if await section.count() > 0:
                company = normalize_text(await section.inner_text())

        # --- списки ---
        tech_stack = normalize_list(await extract_list(page, ["Стек"]))
        responsibilities = normalize_list(await extract_list(page, ["Тебе предстоит"]))
        requirements = normalize_list(await extract_list(page, ["Мы ждем"]))
        nice_to_have = normalize_list(await extract_list(page, ["Будет преимуществом"]))

        vacancy.update({
            "title": title,
            "salary": salary_text,
            "salary_from": salary_from,
            "salary_to": salary_to,
            "location": location,
            "experience": experience,
            "company": company,
            "tech_stack": tech_stack,
            "responsibilities": responsibilities,
            "requirements": requirements,
            "nice_to_have": nice_to_have,
        })

    except Exception as e:
        print(f"Ошибка: {vacancy['url']} → {e}")

    return vacancy


# =========================
# Main
# =========================

async def main():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        print("Собираем список вакансий...")
        vacancies = await get_vacancies(page)

        print(f"Найдено: {len(vacancies)}")

        data = []
        for i, v in enumerate(vacancies, 1):
            print(f"[{i}/{len(vacancies)}] {v['title']}")
            parsed = await parse_vacancy(page, v)
            data.append(parsed)

        await browser.close()

        # --- CSV ---
        keys = [
            "title",
            "category",
            "url",
            "salary",
            "salary_from",
            "salary_to",
            "location",
            "experience",
            "company",
            "tech_stack",
            "responsibilities",
            "requirements",
            "nice_to_have",
        ]

        with open(OUTPUT_FILE, "w", newline="", encoding="utf-8-sig") as f:
            writer = csv.DictWriter(f, fieldnames=keys)
            writer.writeheader()
            writer.writerows(data)

        print(f"\nГотово: {OUTPUT_FILE}")


# запуск (если Jupyter)
await main()

Собираем список вакансий...
Найдено: 20
[1/20] Android-разработчик в Krisha.kz
[2/20] 1С разработчик
[3/20] Middle GO Backend-разработчик Core team
[4/20] Web QA Engineer в Kolesa.kz
[5/20] DevOps-инженер
[6/20] Продуктовый аналитик в Krisha.kz
[7/20] Аналитик-разработчик
[8/20] Продуктовый аналитик в Kolesa.kz
[9/20] Middle UX-исследователь
[10/20] Менеджер по продажам B2B
[11/20] Middle Product manager в Kolesa.kz
[12/20] Product Manager в Krisha.kz
[13/20] Руководитель службы поддержки B2B проектов
[14/20] Менеджер по монетизации рекламы
[15/20] Менеджер по обучению
[16/20] Старший бухгалтер по налогам
[17/20] Видеограф
[18/20] Видеограф
[19/20] Младший менеджер по работе с государственными органами
[20/20] Тимлид отдела телемаркетинга

Готово: kolesa_vacancies.csv
